# 01 — Build the Court case collection

This notebook discovers Court of Justice (`C`) and General Court (`T`) proceedings through complementary InfoCuria searches. It combines four date-sorted searches with two case-number searches, then removes repeated search hits. The result is the starting collection for procedural metadata and document retrieval in notebook 02.

InfoCuria is the source of the search results.


## Inputs and outputs

Input: live InfoCuria search results.

Output: `output/court_infocuria/court_cases_discovered.xlsx` (one row per distinct returned `procedureId`, with any unkeyed search hits retained). The restartable search state lives at `logs/court_infocuria/case_scrape_state.jsonl`.

Run from the repository containing `code/02_court/`, or set `ECCJEU_PROJECT_ROOT`. Notebook 02 reads this workbook.


In [1]:
import os
import re
import json
import time
import random
import logging
from pathlib import Path
from datetime import datetime, date, timedelta

import requests
import pandas as pd
from tqdm.auto import tqdm


## 1. Paths and search settings

The repository has `code/01_commission`, `code/02_court`, and `code/03_mapping`, alongside `data`, `logs`, `input`, and `output`. Court workbooks go directly into `output/court_infocuria/`; JSONL restart state goes into `logs/court_infocuria/`.


In [2]:
# Repository paths

def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "02_court").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
LOG_DIR = PROJECT_ROOT / "logs" / "court_infocuria"

CASE_MANIFEST_XLSX = OUTPUT_DIR / "court_cases_discovered.xlsx"
CASE_STATE_JSONL = LOG_DIR / "case_scrape_state.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Acquisition settings
SEARCH_URL = "https://infocuriaws.curia.europa.eu/elastic-connector/search"
HEADERS = {
    "accept": "application/json",
    "content-type": "application/json; charset=UTF-8",
    "origin": "https://infocuria.curia.europa.eu",
    "referer": "https://infocuria.curia.europa.eu/",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome Safari",
}

START_YEAR = 1950
END_YEAR = 2028
END_YEAR_EXCLUSIVE = END_YEAR + 1
KEEP_PREFIXES = {"C", "T"}
WILDCARD_PREFIXES = ["C", "T"]

PAGE_SIZE = 100
REQUEST_TIMEOUT_SECONDS = 90
SLEEP_BETWEEN_PAGES_SECONDS = 1.0
SLEEP_BETWEEN_WINDOWS_SECONDS = 1.5
SLEEP_BETWEEN_BUCKETS_SECONDS = 1.0
JITTER_SECONDS = 0.7
MAX_HITS_PER_WINDOW = 1500
MIN_WINDOW_DAYS = 1
DEEP_PAGINATION_LIMIT = 10000
DEEP_PAGINATION_WARNING_AT = 9500
CHECKPOINT_EVERY_PASS_OR_BUCKETS = 10
STOP_AFTER_CONSECUTIVE_EMPTY_PAGES = 2
SAVE_RAW_JSON = False
COURT_INFOCURIA_JSON_DIR = LOG_DIR / "raw_responses"

# fresh_scrape: discard the checkpoint and scrape from zero
# update_scrape: keep the checkpoint, scrape again, and merge the latest rows
# rebuild_from_cache: rebuild the final manifest without network requests
SCRAPE_MODE = "update_scrape"
VALID_SCRAPE_MODES = {"fresh_scrape", "update_scrape", "rebuild_from_cache"}
if SCRAPE_MODE not in VALID_SCRAPE_MODES:
    raise ValueError(f"Unknown SCRAPE_MODE: {SCRAPE_MODE}")

DATE_PASSES = [
    {"pass_name": "introductionDate_ASC", "kind": "date", "filter_field": "introductionDate", "sort_term": "INTRODUCTION_DATE", "sort_direction": "ASC"},
    {"pass_name": "introductionDate_DESC", "kind": "date", "filter_field": "introductionDate", "sort_term": "INTRODUCTION_DATE", "sort_direction": "DESC"},
    {"pass_name": "closeDate_ASC", "kind": "date", "filter_field": "closeDate", "sort_term": "CLOSE_DATE", "sort_direction": "ASC"},
    {"pass_name": "closeDate_DESC", "kind": "date", "filter_field": "closeDate", "sort_term": "CLOSE_DATE", "sort_direction": "DESC"},
]
WILDCARD_PASSES = [
    {"pass_name": "case_number_wildcard_ASC", "kind": "wildcard", "sort_term": "AFF_NUM", "sort_direction": "ASC"},
    {"pass_name": "case_number_wildcard_DESC", "kind": "wildcard", "sort_term": "AFF_NUM", "sort_direction": "DESC"},
]
SCRAPE_PASSES = DATE_PASSES + WILDCARD_PASSES

# Human-readable log. The resumable data itself lives in CASE_STATE_JSONL.
log_path = LOG_DIR / "case_scrape.log"
logger = logging.getLogger("court_infocuria_case_scrape")
logger.setLevel(logging.INFO)
logger.handlers.clear()
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

print("Project root:", PROJECT_ROOT)
print("Case manifest:", CASE_MANIFEST_XLSX)
print("Scrape state:", CASE_STATE_JSONL)
print("Mode:", SCRAPE_MODE)


Project root: /home/edik/projects/ec_cjeu
Case manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_cases_discovered.xlsx
Scrape state: /home/edik/projects/ec_cjeu/logs/court_infocuria/case_scrape_state.jsonl
Mode: update_scrape


## 2. Readable case identifiers

InfoCuria provides `affId`, `procedureId`, and a published case number. Keep all three: they describe related but different identifiers and should not be collapsed into a single derived key. The normalized case number helps read and match rows; source identifiers remain unchanged.


In [3]:
# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def polite_sleep(base_seconds: float):
    if base_seconds and base_seconds > 0:
        time.sleep(base_seconds + random.uniform(0, JITTER_SECONDS))


def parse_date_yyyy_mm_dd(s: str) -> date:
    return datetime.strptime(s, "%Y-%m-%d").date()


def fmt_date(d: date) -> str:
    return d.strftime("%Y-%m-%d")


def window_days(start_date: str, end_date: str) -> int:
    return (parse_date_yyyy_mm_dd(end_date) - parse_date_yyyy_mm_dd(start_date)).days


def split_date_window(start_date: str, end_date: str):
    s = parse_date_yyyy_mm_dd(start_date)
    e = parse_date_yyyy_mm_dd(end_date)
    days = (e - s).days
    if days <= 1:
        return None
    mid = s + timedelta(days=days // 2)
    return (fmt_date(s), fmt_date(mid)), (fmt_date(mid), fmt_date(e))


def yy_from_year(year: int) -> str:
    return f"{year % 100:02d}"


def bucket_search_term(prefix: str, year: int) -> str:
    return f"{prefix}-*/{yy_from_year(year)}"


def get_ml_value(ml_list, lang="en"):
    if not isinstance(ml_list, list):
        return None
    for item in ml_list:
        if isinstance(item, dict) and lang in item:
            return item[lang]
    return None


def extract_ml_labels(items, lang="en"):
    if not isinstance(items, list):
        return None
    labels = []
    for item in items:
        if not isinstance(item, dict):
            continue
        code = item.get("code")
        label = get_ml_value(item.get("label", []), lang)
        if code and label:
            labels.append(f"{code}: {label}")
        elif code:
            labels.append(str(code))
        elif label:
            labels.append(str(label))
    return " | ".join(labels) if labels else None


def first_nonempty(*values):
    for v in values:
        if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != "":
            return v
    return None
# ---------------------------------------------------------------------
# Clean case-number extraction
# ---------------------------------------------------------------------

CASE_NUMBER_RE = re.compile(
    r"""
    (?P<prefix>[A-Z]{1,4})
    \s*[-–—]?\s*
    (?P<number>\d{1,4})
    \s*[/\-]\s*
    (?P<year>\d{2,4})
    (?P<suffix>(?:\s+[A-Z][A-Z0-9]*(?:[-_/][A-Z0-9]+)*)*)
    """,
    re.VERBOSE,
)


def normalize_year(year_text):
    if year_text is None:
        return None
    y = str(year_text).strip()
    if not y.isdigit():
        return None
    if len(y) == 4:
        return int(y)
    if len(y) == 2:
        yy = int(y)
        return 1900 + yy if yy >= 50 else 2000 + yy
    return None


def normalize_suffix(suffix_text):
    """Return a compact case suffix such as P, P-DEP, SA, PPU, R, or empty string."""
    if suffix_text is None:
        return ""
    suffix = str(suffix_text).strip().upper()
    suffix = suffix.replace("–", "-").replace("—", "-")
    suffix = re.sub(r"\s+", "-", suffix)
    suffix = suffix.strip("-/")
    return suffix


def make_internal_key(year_full, case_number_padded, case_prefix, case_suffix=""):
    """Build YYYY/NNNN/PREFIX/SUFFIX/; the suffix slot is kept even when suffix is empty."""
    if year_full is None or pd.isna(year_full):
        return None
    if case_number_padded is None or pd.isna(case_number_padded) or str(case_number_padded).strip() == "":
        return None
    if case_prefix is None or pd.isna(case_prefix) or str(case_prefix).strip() == "":
        return None
    year = int(float(year_full))
    number = str(case_number_padded).strip().zfill(4)
    prefix = str(case_prefix).strip().upper()
    suffix = normalize_suffix(case_suffix)
    return f"{year:04d}/{number}/{prefix}/{suffix}/"


def clean_case_number(raw):
    text = "" if raw is None or (isinstance(raw, float) and pd.isna(raw)) else str(raw).strip()
    if not text:
        return {
            "case_number_raw": None,
            "case_number_clean": None,
            "case_prefix": None,
            "case_suffix": "",
            "case_number_int": None,
            "case_number_padded": None,
            "case_year_raw": None,
            "case_year_full": None,
            "internal_key": None,
            "case_number_clean_status": "missing_raw_case_number",
        }

    text_norm = text.upper().replace("–", "-").replace("—", "-")
    match = CASE_NUMBER_RE.search(text_norm)
    if not match:
        return {
            "case_number_raw": text,
            "case_number_clean": None,
            "case_prefix": None,
            "case_suffix": "",
            "case_number_int": None,
            "case_number_padded": None,
            "case_year_raw": None,
            "case_year_full": None,
            "internal_key": None,
            "case_number_clean_status": "no_case_number_pattern",
        }

    prefix = match.group("prefix")
    number_int = int(match.group("number"))
    number_padded = f"{number_int:04d}"
    year_raw = match.group("year")
    year_full = normalize_year(year_raw)
    year_2 = f"{year_full % 100:02d}" if year_full is not None else year_raw[-2:]
    case_suffix = normalize_suffix(match.group("suffix"))
    clean = f"{prefix}-{number_int}/{year_2}" + (f" {case_suffix}" if case_suffix else "")
    internal_key = make_internal_key(year_full, number_padded, prefix, case_suffix)

    return {
        "case_number_raw": text,
        "case_number_clean": clean,
        "case_prefix": prefix,
        "case_suffix": case_suffix,
        "case_number_int": number_int,
        "case_number_padded": number_padded,
        "case_year_raw": year_raw,
        "case_year_full": year_full,
        "internal_key": internal_key,
        "case_number_clean_status": "ok",
    }


def add_case_key_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add/refresh case_prefix, case_suffix, padded number, year, and internal_key columns from case_number_raw."""
    if df.empty:
        return df.copy()
    out = df.copy()
    source_col = "case_number_raw" if "case_number_raw" in out.columns else None
    if source_col is None:
        for candidate in ["publishedId", "publishedAffId", "affId", "case_number_clean"]:
            if candidate in out.columns:
                source_col = candidate
                break
    if source_col is None:
        raise KeyError("Cannot add case key columns: no case-number-like column found.")

    parsed = out[source_col].apply(clean_case_number).apply(pd.Series)
    for col in parsed.columns:
        if col == "case_number_raw" and "case_number_raw" in out.columns:
            continue
        out[col] = parsed[col]
    if "case_number_raw" not in out.columns:
        out["case_number_raw"] = parsed["case_number_raw"]
    return out


def is_relevant_prefix(row_or_dict) -> bool:
    prefix = row_or_dict.get("case_prefix")
    return prefix in KEEP_PREFIXES


## 3. Find the Court collection

| Search route | What it contributes |
| --- | --- |
| Introduction date, ascending and descending | Proceedings found through their introduction date |
| Closing date, ascending and descending | Proceedings found through their closing date |
| `C-*/YY` and `T-*/YY`, ascending and descending | Proceedings found by published case number |

The union of these routes improves coverage when result ordering or individual queries vary. Large date windows are split to avoid the endpoint’s pagination limit. Only `C` and `T` records are retained. The JSONL state supports rebuilding the workbook without repeating network requests.


In [4]:
# ---------------------------------------------------------------------
# InfoCuria API calls
# ---------------------------------------------------------------------

def make_date_payload(page_number=0, page_size=PAGE_SIZE, start_date=None, end_date=None, filter_field="introductionDate", sort_term="INTRODUCTION_DATE", sort_direction="ASC"):
    from_ = page_number * page_size + 1
    to_ = from_ + page_size - 1
    payload = {
        "searchTerm": "",
        "multiSearchTerms": [],
        "sortTermList": [{"sortDirection": sort_direction, "sortTerm": sort_term, "sortSourceTab": "affair"}],
        "pagination": {"pageNumber": page_number, "pageSize": page_size, "from": from_, "to": to_},
        "language": "EN",
        "tabName": "affair",
        "isAllTabsRequest": False,
        "ecli": "",
        "publishedId": "",
        "usualName": "",
        "logicDocId": "",
        "isSearchExact": True,
        "searchSources": ["document", "metadata"],
    }
    if start_date and end_date and filter_field:
        payload["filtersValue"] = [{"field": filter_field, "values": [start_date, end_date], "valuesWithFullHierarchy": [filter_field]}]
    return payload


def make_wildcard_payload(search_term: str, page_number=0, page_size=PAGE_SIZE, sort_direction="ASC"):
    from_ = page_number * page_size + 1
    to_ = from_ + page_size - 1
    return {
        "searchTerm": search_term,
        "multiSearchTerms": [],
        "sortTermList": [{"sortDirection": sort_direction, "sortTerm": "AFF_NUM", "sortSourceTab": "affair"}],
        "pagination": {"pageNumber": page_number, "pageSize": page_size, "from": from_, "to": to_},
        "language": "EN",
        "tabName": "affair",
        "isAllTabsRequest": True,
        "ecli": "",
        "publishedId": "",
        "usualName": "",
        "logicDocId": "",
        "isSearchExact": True,
        "searchSources": ["document", "metadata"],
    }


def post_json(payload, max_retries=6):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(SEARCH_URL, headers=HEADERS, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.ok:
                return response.json()
            last_error = f"HTTP {response.status_code}: {response.text[:500]}"
        except Exception as exc:
            last_error = repr(exc)

        wait = min(90, 2 ** attempt) + random.uniform(0, 1.5)
        logger.warning("Request failed attempt %s/%s: %s | sleeping %.1fs", attempt, max_retries, last_error, wait)
        time.sleep(wait)

    raise RuntimeError(f"InfoCuria request failed after {max_retries} retries: {last_error}")


def fetch_date_page(page_number=0, page_size=PAGE_SIZE, **kwargs):
    return post_json(make_date_payload(page_number=page_number, page_size=page_size, **kwargs))


def fetch_wildcard_page(search_term: str, page_number=0, page_size=PAGE_SIZE, sort_direction="ASC"):
    return post_json(make_wildcard_payload(search_term=search_term, page_number=page_number, page_size=page_size, sort_direction=sort_direction))

print("Date payload sample:")
print(json.dumps(make_date_payload(page_number=0, page_size=20, start_date="2025-01-01", end_date="2026-01-01"), indent=2, ensure_ascii=False))
print("\nWildcard payload sample:")
print(json.dumps(make_wildcard_payload("C-*/25", page_number=0, page_size=20), indent=2, ensure_ascii=False))
# ---------------------------------------------------------------------
# Flatten search hits
# ---------------------------------------------------------------------

def flatten_case_hit(hit, scrape_pass=None, scrape_kind=None, scrape_window_start=None, scrape_window_end=None, page_number=None, position_on_page=None, bucket_prefix=None, bucket_year=None, bucket_search_term=None, sort_direction=None):
    c = hit.get("content", hit)

    raw_case = first_nonempty(c.get("publishedId"), c.get("publishedAffId"), c.get("affaire"), c.get("caseNumber"))
    clean = clean_case_number(raw_case)

    row = {
        **clean,
        "scrape_pass": scrape_pass,
        "scrape_kind": scrape_kind,
        "scrape_window_start": scrape_window_start,
        "scrape_window_end": scrape_window_end,
        "bucket_prefix": bucket_prefix,
        "bucket_year": bucket_year,
        "bucket_year_yy": yy_from_year(int(bucket_year)) if bucket_year is not None else None,
        "bucket_search_term": bucket_search_term,
        "scrape_sort_direction": sort_direction,
        "scrape_page_number": page_number,
        "scrape_position_on_page": position_on_page,
        "affId": c.get("affId"),
        "publishedId": c.get("publishedId"),
        "publishedAffId": c.get("publishedAffId"),
        "procedureId": c.get("procedureId"),
        "case_name_en": get_ml_value(c.get("usualNameML"), "en"),
        "case_name_de": get_ml_value(c.get("usualNameML"), "de"),
        "jurisdictionCode": c.get("jurisdictionCode"),
        "natureCode": c.get("natureCode"),
        "introductionDate": c.get("introductionDate"),
        "introductionYear": c.get("introductionYear"),
        "closeDate": c.get("closeDate"),
        "closeYear": c.get("closeYear"),
        "affairStateCode": c.get("affairStateCode"),
        "matCode": c.get("matCode"),
        "subject_matter_text": extract_ml_labels(c.get("matCodeML"), "en"),
        "procLang": c.get("procLang"),
        "ecli": c.get("ecli"),
        "logicDocId": c.get("logicDocId"),
        "joinExist": c.get("joinExist"),
        "joinAffairs": json.dumps(c.get("joinAffairs"), ensure_ascii=False) if c.get("joinAffairs") is not None else None,
        "raw_hit_type": hit.get("type"),
        "raw_content_json": json.dumps(c, ensure_ascii=False, sort_keys=True),
    }
    return row
# ---------------------------------------------------------------------
# Date-window scraping
# ---------------------------------------------------------------------

def fetch_cases_for_date_window(start_date: str, end_date: str, pass_cfg: dict, depth=0):
    kwargs = {
        "filter_field": pass_cfg["filter_field"],
        "sort_term": pass_cfg["sort_term"],
        "sort_direction": pass_cfg["sort_direction"],
    }
    pass_name = pass_cfg["pass_name"]
    first_page = fetch_date_page(page_number=0, page_size=PAGE_SIZE, start_date=start_date, end_date=end_date, **kwargs)
    total_hits = int(first_page.get("totalHits", 0) or 0)
    days = window_days(start_date, end_date)

    logger.info("%s | %s to %s | hits=%s | days=%s | depth=%s", pass_name, start_date, end_date, total_hits, days, depth)
    print(f"{pass_name}: {start_date} to {end_date} | hits={total_hits:,} | days={days} | depth={depth}", flush=True)

    if total_hits == 0:
        polite_sleep(SLEEP_BETWEEN_WINDOWS_SECONDS)
        return [], []

    # Split before deep pagination becomes a problem.
    if (total_hits > MAX_HITS_PER_WINDOW or total_hits >= DEEP_PAGINATION_WARNING_AT) and days > MIN_WINDOW_DAYS:
        split = split_date_window(start_date, end_date)
        if split is not None:
            left, right = split
            left_rows, left_log = fetch_cases_for_date_window(left[0], left[1], pass_cfg, depth=depth + 1)
            right_rows, right_log = fetch_cases_for_date_window(right[0], right[1], pass_cfg, depth=depth + 1)
            return left_rows + right_rows, left_log + right_log

    n_pages = (total_hits + PAGE_SIZE - 1) // PAGE_SIZE
    rows = []
    page_log = []

    if total_hits >= DEEP_PAGINATION_WARNING_AT:
        logger.warning("%s %s-%s has %s hits and may hit deep pagination.", pass_name, start_date, end_date, total_hits)

    for page_number in range(n_pages):
        requested_from = page_number * PAGE_SIZE + 1
        requested_to = requested_from + PAGE_SIZE - 1
        if requested_from > DEEP_PAGINATION_LIMIT:
            logger.warning("Stopping %s %s-%s before deep pagination from=%s", pass_name, start_date, end_date, requested_from)
            break

        if page_number == 0:
            data = first_page
        else:
            data = fetch_date_page(page_number=page_number, page_size=PAGE_SIZE, start_date=start_date, end_date=end_date, **kwargs)

        hits = data.get("searchHits", []) or []
        kept = 0
        for i, hit in enumerate(hits):
            row = flatten_case_hit(hit, scrape_pass=pass_name, scrape_kind="date", scrape_window_start=start_date, scrape_window_end=end_date, page_number=page_number, position_on_page=i, sort_direction=pass_cfg["sort_direction"])
            if is_relevant_prefix(row):
                rows.append(row)
                kept += 1

        page_log.append({
            "scrape_pass": pass_name,
            "scrape_kind": "date",
            "window_start": start_date,
            "window_end": end_date,
            "page_number": page_number,
            "requested_from": requested_from,
            "requested_to": requested_to,
            "totalHits_reported": int(data.get("totalHits", 0) or 0),
            "hits_returned": len(hits),
            "rows_kept_prefix_CT": kept,
        })
        polite_sleep(SLEEP_BETWEEN_PAGES_SECONDS)

    polite_sleep(SLEEP_BETWEEN_WINDOWS_SECONDS)
    return rows, page_log


def run_date_pass(pass_cfg: dict):
    all_rows = []
    all_page_log = []
    for year in tqdm(range(START_YEAR, END_YEAR_EXCLUSIVE), desc=f"Scraping {pass_cfg['pass_name']}"):
        start_date = f"{year}-01-01"
        end_date = f"{year + 1}-01-01"
        try:
            rows, page_log = fetch_cases_for_date_window(start_date, end_date, pass_cfg)
            all_rows.extend(rows)
            all_page_log.extend(page_log)
        except Exception as exc:
            logger.exception("Failed %s year %s: %s", pass_cfg["pass_name"], year, exc)
            print(f"FAILED {pass_cfg['pass_name']} {year}: {exc}", flush=True)
            all_page_log.append({"scrape_pass": pass_cfg["pass_name"], "scrape_kind": "date", "window_start": start_date, "window_end": end_date, "error": repr(exc)})
    return all_rows, all_page_log
# ---------------------------------------------------------------------
# Prefix/year wildcard scraping: C-*/YY and T-*/YY
# ---------------------------------------------------------------------

def run_wildcard_bucket(prefix: str, year: int, pass_cfg: dict):
    search_term = bucket_search_term(prefix, year)
    pass_name = pass_cfg["pass_name"]
    sort_direction = pass_cfg["sort_direction"]

    rows = []
    page_log = []

    first_page = fetch_wildcard_page(search_term=search_term, page_number=0, page_size=PAGE_SIZE, sort_direction=sort_direction)
    total_hits = int(first_page.get("totalHits", 0) or 0)
    n_pages = (total_hits + PAGE_SIZE - 1) // PAGE_SIZE

    logger.info("%s | bucket=%s | totalHits=%s | planned pages=%s", pass_name, search_term, total_hits, n_pages)

    if total_hits >= DEEP_PAGINATION_WARNING_AT:
        logger.warning("Bucket %s/%s has %s hits; this may hit deep pagination.", pass_name, search_term, total_hits)
        print(f"WARNING: {pass_name} {search_term} has {total_hits:,} hits; may hit deep pagination.")

    consecutive_empty = 0
    seen_page_fingerprints = set()

    for page_number in range(n_pages):
        requested_from = page_number * PAGE_SIZE + 1
        requested_to = requested_from + PAGE_SIZE - 1
        if requested_from > DEEP_PAGINATION_LIMIT:
            logger.warning("Stopping bucket %s %s before deep pagination from=%s", pass_name, search_term, requested_from)
            break

        if page_number == 0:
            data = first_page
        else:
            data = fetch_wildcard_page(search_term=search_term, page_number=page_number, page_size=PAGE_SIZE, sort_direction=sort_direction)

        if SAVE_RAW_JSON:
            raw_path = COURT_INFOCURIA_JSON_DIR / f"{pass_name}_{prefix}_{year}_page_{page_number:05d}.json"
            raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

        hits = data.get("searchHits", []) or []
        page_ids = []
        kept = 0
        off_bucket = 0

        for i, hit in enumerate(hits):
            c = hit.get("content", hit)
            page_ids.append(str(first_nonempty(c.get("affId"), c.get("publishedId"), c.get("procedureId"), i)))
            row = flatten_case_hit(hit, scrape_pass=pass_name, scrape_kind="wildcard", bucket_prefix=prefix, bucket_year=year, bucket_search_term=search_term, page_number=page_number, position_on_page=i, sort_direction=sort_direction)
            expected_year = year
            expected_prefix = prefix
            matches_bucket = (row.get("case_prefix") == expected_prefix and row.get("case_year_full") == expected_year)
            row["matches_bucket_prefix_year"] = matches_bucket
            if not matches_bucket:
                off_bucket += 1
            if is_relevant_prefix(row):
                rows.append(row)
                kept += 1

        fingerprint = tuple(page_ids)
        repeated_page = fingerprint in seen_page_fingerprints and bool(fingerprint)
        seen_page_fingerprints.add(fingerprint)

        page_log.append({
            "scrape_pass": pass_name,
            "scrape_kind": "wildcard",
            "bucket_prefix": prefix,
            "bucket_year": year,
            "bucket_year_yy": yy_from_year(year),
            "bucket_search_term": search_term,
            "page_number": page_number,
            "requested_from": requested_from,
            "requested_to": requested_to,
            "totalHits_reported": int(data.get("totalHits", 0) or 0),
            "hits_returned": len(hits),
            "rows_kept_prefix_CT": kept,
            "off_bucket_rows_on_page": off_bucket,
            "first_id_on_page": page_ids[0] if page_ids else None,
            "last_id_on_page": page_ids[-1] if page_ids else None,
            "repeated_page_fingerprint": repeated_page,
        })

        if len(hits) == 0:
            consecutive_empty += 1
        else:
            consecutive_empty = 0

        if repeated_page:
            logger.warning("Repeated page fingerprint in %s %s page %s. Stopping bucket.", pass_name, search_term, page_number)
            break

        if consecutive_empty >= STOP_AFTER_CONSECUTIVE_EMPTY_PAGES:
            logger.warning("%s consecutive empty pages in %s %s page %s. Stopping bucket.", consecutive_empty, pass_name, search_term, page_number)
            break

        polite_sleep(SLEEP_BETWEEN_PAGES_SECONDS)

    polite_sleep(SLEEP_BETWEEN_BUCKETS_SECONDS)
    return rows, page_log


def run_wildcard_pass(pass_cfg: dict):
    all_rows = []
    all_page_log = []
    buckets = [(prefix, year) for prefix in WILDCARD_PREFIXES for year in range(START_YEAR, END_YEAR + 1)]
    print(f"{pass_cfg['pass_name']} planned buckets: {len(buckets):,}")

    for idx, (prefix, year) in enumerate(tqdm(buckets, desc=f"Scraping {pass_cfg['pass_name']}"), start=1):
        search_term = bucket_search_term(prefix, year)
        try:
            rows, page_log = run_wildcard_bucket(prefix, year, pass_cfg)
            all_rows.extend(rows)
            all_page_log.extend(page_log)
        except Exception as exc:
            logger.exception("Wildcard bucket failed %s %s: %s", pass_cfg["pass_name"], search_term, exc)
            print(f"FAILED {pass_cfg['pass_name']} {search_term}: {exc}", flush=True)
            all_page_log.append({"scrape_pass": pass_cfg["pass_name"], "scrape_kind": "wildcard", "bucket_prefix": prefix, "bucket_year": year, "bucket_search_term": search_term, "error": repr(exc)})

        if idx % CHECKPOINT_EVERY_PASS_OR_BUCKETS == 0:
            print(f"{pass_cfg['pass_name']} checkpoint bucket {idx}/{len(buckets)}: raw kept rows so far={len(all_rows):,}")
    return all_rows, all_page_log


Date payload sample:
{
  "searchTerm": "",
  "multiSearchTerms": [],
  "sortTermList": [
    {
      "sortDirection": "ASC",
      "sortTerm": "INTRODUCTION_DATE",
      "sortSourceTab": "affair"
    }
  ],
  "pagination": {
    "pageNumber": 0,
    "pageSize": 20,
    "from": 1,
    "to": 20
  },
  "language": "EN",
  "tabName": "affair",
  "isAllTabsRequest": false,
  "ecli": "",
  "publishedId": "",
  "usualName": "",
  "logicDocId": "",
  "isSearchExact": true,
  "searchSources": [
    "document",
    "metadata"
  ],
  "filtersValue": [
    {
      "field": "introductionDate",
      "values": [
        "2025-01-01",
        "2026-01-01"
      ],
      "valuesWithFullHierarchy": [
        "introductionDate"
      ]
    }
  ]
}

Wildcard payload sample:
{
  "searchTerm": "C-*/25",
  "multiSearchTerms": [],
  "sortTermList": [
    {
      "sortDirection": "ASC",
      "sortTerm": "AFF_NUM",
      "sortSourceTab": "affair"
    }
  ],
  "pagination": {
    "pageNumber": 0,
    "pageSize

## 4. Combine search hits

A search can return the same procedure several times. Keep one row for each non-empty `procedureId`; retain records missing that identifier for inspection rather than treating them as an arbitrary match. `affId` and the published case number remain in the workbook. Repeated hits and search-route details stay in the restart state.


In [5]:
MANIFEST_COLUMNS = [
    "case_number_raw", "case_number_clean", "case_prefix", "case_suffix",
    "case_number_int", "case_number_padded", "case_year_raw", "case_year_full",
    "internal_key", "affId", "publishedId", "publishedAffId", "procedureId",
    "case_name_en", "jurisdictionCode", "natureCode", "introductionDate",
    "closeDate", "affairStateCode", "subject_matter_text", "procLang", "ecli",
    "joinExist", "joinAffairs",
]


def choose_dedup_key(row):
    value = row.get("procedureId")
    if pd.notna(value) and str(value).strip():
        return f"procedureId:{str(value).strip()}"
    return None


def deduplicate_cases(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw.empty:
        return df_raw.copy()

    df = add_case_key_columns(df_raw)
    df["dedup_key"] = df.apply(choose_dedup_key, axis=1)
    keyed = df[df["dedup_key"].notna()].copy()
    unkeyed = df[df["dedup_key"].isna()].drop_duplicates().copy()

    pass_order = {cfg["pass_name"]: i for i, cfg in enumerate(SCRAPE_PASSES)}
    if not keyed.empty:
        keyed["_pass_order"] = keyed["scrape_pass"].map(pass_order).fillna(999)
        keyed["_source_order"] = pd.to_numeric(keyed.get("raw_source_priority", 999), errors="coerce").fillna(999)
        keyed = keyed.sort_values(
            [c for c in ["dedup_key", "_source_order", "_pass_order", "scrape_window_start", "bucket_year", "scrape_page_number"] if c in keyed.columns],
            na_position="last",
        )
        keyed = keyed.drop_duplicates("dedup_key", keep="first")
        keyed = keyed.drop(columns=["_pass_order", "_source_order"], errors="ignore")

    out = pd.concat([keyed, unkeyed], ignore_index=True, sort=False)
    out = add_case_key_columns(out)
    for column in MANIFEST_COLUMNS:
        if column not in out.columns:
            out[column] = pd.NA
    return out[MANIFEST_COLUMNS].reset_index(drop=True)


def read_state(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_json(path, orient="records", lines=True, dtype=False)


def save_state(df: pd.DataFrame) -> None:
    CASE_STATE_JSONL.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(CASE_STATE_JSONL, orient="records", lines=True, force_ascii=False)


def save_manifest(df_raw: pd.DataFrame) -> pd.DataFrame:
    manifest = deduplicate_cases(df_raw)
    manifest.to_excel(CASE_MANIFEST_XLSX, index=False, sheet_name="cases")
    return manifest


def scrape_all_passes() -> pd.DataFrame:
    all_rows = []
    for pass_cfg in SCRAPE_PASSES:
        print(f"\nRunning {pass_cfg['pass_name']}")
        if pass_cfg["kind"] == "date":
            rows, _ = run_date_pass(pass_cfg)
        else:
            rows, _ = run_wildcard_pass(pass_cfg)

        loaded_at = datetime.now().isoformat(timespec="seconds")
        for row in rows:
            row["raw_source"] = "fresh_scrape"
            row["raw_source_priority"] = 0
            row["raw_loaded_at"] = loaded_at
        all_rows.extend(rows)
        save_state(pd.DataFrame(all_rows))
        print(f"Rows retained in checkpoint: {len(all_rows):,}")
    return pd.DataFrame(all_rows)


def build_manifest():
    if SCRAPE_MODE == "fresh_scrape":
        if CASE_STATE_JSONL.exists():
            CASE_STATE_JSONL.unlink()
        raw = scrape_all_passes()

    elif SCRAPE_MODE == "update_scrape":
        cached = read_state(CASE_STATE_JSONL)
        if not cached.empty:
            cached["raw_source_priority"] = 1
        new = scrape_all_passes()
        if not new.empty:
            new["raw_source_priority"] = 0
        raw = pd.concat([new, cached], ignore_index=True, sort=False)
        identity = [c for c in ["affId", "publishedId", "publishedAffId", "procedureId", "case_number_raw", "scrape_pass"] if c in raw.columns]
        if identity:
            raw = raw.sort_values("raw_source_priority", na_position="last").drop_duplicates(identity, keep="first")
        save_state(raw)

    elif SCRAPE_MODE == "rebuild_from_cache":
        raw = read_state(CASE_STATE_JSONL)
        if raw.empty:
            raise FileNotFoundError(f"No scrape checkpoint found: {CASE_STATE_JSONL}")

    else:
        raise ValueError(SCRAPE_MODE)

    manifest = save_manifest(raw)
    return manifest, raw


df_cases, df_raw_cases = build_manifest()
print(f"Final procedure rows: {len(df_cases):,}")



Running introductionDate_ASC


Scraping introductionDate_ASC:   0%|          | 0/79 [00:00<?, ?it/s]

INFO | introductionDate_ASC | 1950-01-01 to 1951-01-01 | hits=0 | days=365 | depth=0


introductionDate_ASC: 1950-01-01 to 1951-01-01 | hits=0 | days=365 | depth=0


INFO | introductionDate_ASC | 1951-01-01 to 1952-01-01 | hits=0 | days=365 | depth=0


...

case_number_wildcard_DESC checkpoint bucket 140/158: raw kept rows so far=34,593


INFO | case_number_wildcard_DESC | bucket=T-*/11 | totalHits=676 | planned pages=7
INFO | case_number_wildcard_DESC | bucket=T-*/12 | totalHits=593 | planned pages=6
INFO | case_number_wildcard_DESC | bucket=T-*/13 | totalHits=757 | planned pages=8
INFO | case_number_wildcard_DESC | bucket=T-*/14 | totalHits=990 | planned pages=10
INFO | case_number_wildcard_DESC | bucket=T-*/15 | totalHits=782 | planned pages=8
INFO | case_number_wildcard_DESC | bucket=T-*/16 | totalHits=922 | planned pages=10
INFO | case_number_wildcard_DESC | bucket=T-*/17 | totalHits=875 | planned pages=9
INFO | case_number_wildcard_DESC | bucket=T-*/18 | totalHits=793 | planned pages=8
INFO | case_number_wildcard_DESC | bucket=T-*/19 | totalHits=971 | planned pages=10
INFO | case_number_wildcard_DESC | bucket=T-*/20 | totalHits=837 | planned pages=9


case_number_wildcard_DESC checkpoint bucket 150/158: raw kept rows so far=42,789


INFO | case_number_wildcard_DESC | bucket=T-*/21 | totalHits=895 | planned pages=9
INFO | case_number_wildcard_DESC | bucket=T-*/22 | totalHits=897 | planned pages=9
INFO | case_number_wildcard_DESC | bucket=T-*/23 | totalHits=1255 | planned pages=13
INFO | case_number_wildcard_DESC | bucket=T-*/24 | totalHits=736 | planned pages=8
INFO | case_number_wildcard_DESC | bucket=T-*/25 | totalHits=959 | planned pages=10
INFO | case_number_wildcard_DESC | bucket=T-*/26 | totalHits=570 | planned pages=6
INFO | case_number_wildcard_DESC | bucket=T-*/27 | totalHits=0 | planned pages=0
INFO | case_number_wildcard_DESC | bucket=T-*/28 | totalHits=0 | planned pages=0


Rows retained in checkpoint: 319,846
Final procedure rows: 57,413


## 5. Check the collection

Report the number of rows, missing identifiers, distinct `affId` and `procedureId` values, and the jurisdiction prefixes. These describe the collection obtained by the searches; they do not by themselves establish complete historical coverage.


In [6]:
nonempty_procedure = df_cases["procedureId"].fillna("").astype(str).str.strip().ne("")
nonempty_affair = df_cases["affId"].fillna("").astype(str).str.strip().ne("")
print(f"Discovered rows: {len(df_cases):,}")
print("Distinct procedureId:", df_cases.loc[nonempty_procedure, "procedureId"].nunique())
print("Distinct affId:", df_cases.loc[nonempty_affair, "affId"].nunique())
print("Rows without procedureId:", int((~nonempty_procedure).sum()))
print("Rows without affId:", int((~nonempty_affair).sum()))
print("Duplicate non-empty procedureId:", int(df_cases.loc[nonempty_procedure, "procedureId"].duplicated().sum()))
print("Rows without normalized case number:", int(df_cases["case_number_clean"].isna().sum()))
display(df_cases["case_prefix"].value_counts(dropna=False).rename_axis("case_prefix").reset_index(name="rows"))
print("Saved:", CASE_MANIFEST_XLSX)


Discovered rows: 57,413
Distinct procedureId: 57413
Distinct affId: 52207
Rows without procedureId: 0
Rows without affId: 0
Duplicate non-empty procedureId: 0
Rows without normalized case number: 0


,case_prefix,rows
0,C,30659
1,T,26754


Saved: /home/edik/projects/ec_cjeu/output/court_infocuria/court_cases_discovered.xlsx
